# 04 — Phase Trajectory Conditioning

**역할**: per-step phase trajectory `(cos φₜ, sin φₜ)`를 U-Net residual block에 **FiLM** 방식으로 주입하는 main trajectory 모델을 60 epoch 학습 + offline phase sensitivity 시각화.

**산출물**: `checkpoints/phase_trajectory_ckpt.pt`, `figures/phase_trajectory_loss.png`, `figures/phase_trajectory_sensitivity.png`

**예상 소요**: **~25분**

---

이 노트북은 **실행 orchestration만 담당**합니다. 핵심 로직은 `pcdp/configs.py`(`phase_trajectory` config), `pcdp/experiment_runner.py`(train-or-load wrapper + `sample_frequency_variants`), `pcdp/models.py`(`PhaseConditionedUnet1D` + per-step FiLM block), `pcdp/training.py`(DDPM loop + `trajectory_phase_cond_fn`), `pcdp/sampling.py`(`sample_action_chunk` + `trajectory_phase_sample_cond_fn`), `pcdp/phase.py`(`make_phase_trajectory_fn`), `pcdp/evaluation.py`(`build_frequency_sweep_protocol`), `pcdp/experiment_plots.py`(loss + sensitivity plots)에 있습니다.

In [ ]:
# Google Drive mount and project-root setup for Colab
from pathlib import Path
import os

PROJECT_DIR = Path('/content/drive/MyDrive/phase_conditioned_diffusion_policy')

try:
    from google.colab import drive
except ImportError:
    print(f'Not running in Google Colab; keeping current working directory: {Path.cwd()}')
else:
    drive.mount('/content/drive')
    if not PROJECT_DIR.exists():
        raise FileNotFoundError(
            f'Expected project directory not found: {PROJECT_DIR}\n'
            'Update PROJECT_DIR to the Google Drive folder that contains this repository.'
        )
    os.chdir(PROJECT_DIR)
    print(f'Current working directory: {Path.cwd()}')


In [ ]:
# Colab dependency setup
import importlib.util
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("google.colab") is None:
    print("Not running in Google Colab; skipping dependency installation and using the current environment.")
else:
    project_root = Path.cwd()
    requirements_path = project_root / "requirements.txt"
    install_command = [sys.executable, "-m", "pip", "install"]

    if requirements_path.exists():
        # requirements.txt pins the runtime stack; -e . installs this repo from pyproject.toml.
        install_command.extend(["-r", str(requirements_path), "-e", str(project_root)])
    else:
        # Fallback to pyproject.toml dependencies if requirements.txt is unavailable.
        install_command.extend(["-e", str(project_root)])

    subprocess.check_call(install_command)

    import gymnasium as gym
    import mujoco
    import minari
    import torch

    gym.make("Ant-v5").close()

    print(f"gymnasium={gym.__version__}")
    print(f"mujoco={mujoco.__version__}")
    print(f"minari={minari.__version__}")
    print(f"torch={torch.__version__}")
    print("Ant-v5 environment smoke check passed.")


## 1. Artifact paths


In [ ]:
from pcdp.paths import ARTIFACT_ROOT, DATA_DIR, CHECKPOINTS_DIR, FIGURES_DIR, ensure_artifact_dirs
ensure_artifact_dirs()
print(f'✓ artifact root: {ARTIFACT_ROOT}')


## 2. Imports + Config

In [ ]:
import torch

from pcdp.configs import get_experiment_config
from pcdp.experiment_plots import plot_action_chunks, plot_loss_curve
from pcdp.experiment_runner import (
    build_model,
    build_noise_scheduler,
    load_data_and_build_loaders,
    sample_frequency_variants,
    train_or_load_checkpoint,
)
from pcdp.evaluation import build_frequency_sweep_protocol
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}, device={device}')

cfg = get_experiment_config('phase_trajectory')
train_cond_fn = cfg.resolve_train_cond_fn()
sample_cond_fn = cfg.resolve_sample_cond_fn()
print(f'Experiment config: {cfg.name} — {cfg.display_name}')


## 3. Data

In [ ]:
data, train_ds, val_ds, train_loader, val_loader = load_data_and_build_loaders(cfg, DATA_DIR)


## 4. Model + Scheduler

In [ ]:
model = build_model(cfg, data, device=device)
noise_scheduler, ns_config, NUM_INFERENCE_STEPS = build_noise_scheduler(cfg)
ema = cfg.build_ema(model)


## 5. Train or Load

In [ ]:
TRAIN = True
train_losses, val_log, best_ema_state, CKPT_PATH = train_or_load_checkpoint(
    train=TRAIN, cfg=cfg, model=model, ema=ema, noise_scheduler=noise_scheduler,
    train_loader=train_loader, val_loader=val_loader, checkpoints_dir=CHECKPOINTS_DIR, device=device,
)


## 6. Loss Curve

In [ ]:
plot_loss_curve(train_losses, val_log, FIGURES_DIR / cfg.artifacts.loss_plot_name, title=cfg.display_name)


## 7. Offline Phase-Trajectory Sensitivity

In [ ]:
freq_protocol = build_frequency_sweep_protocol(data, n_in_dist=3, ood_iqr_scale=1.5)
freqs = freq_protocol.sweep_freqs.tolist()
labels = freq_protocol.zone_labels.tolist()
ep_obs = val_ds[0]['obs'].unsqueeze(0)
samples_by_freq = sample_frequency_variants(
    model, ema, ns_config, ep_obs, data, sample_cond_fn, freqs, labels,
    device=device, num_inference_steps=NUM_INFERENCE_STEPS, dt=cfg.evaluation.dt, seed=data['seed'],
    reference_label=labels[2],
)
plot_action_chunks(
    samples_by_freq, FIGURES_DIR / 'phase_trajectory_sensitivity.png',
    title='Trajectory DP — same obs, different phase trajectories', act_dim=data['ACT_DIM'],
)
